<a href="https://colab.research.google.com/github/gavrilovAlikhan/ML-Practice/blob/main/Student%20Scores/student_scores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [0]:
%pip install catboost lightgbm optuna

## Libraries

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder, OneHotEncoder, StandardScaler, MinMaxScaler

from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.base import clone

from sklearn.model_selection import StratifiedKFold, KFold,cross_validate, cross_val_score, cross_val_predict
from sklearn.metrics import root_mean_squared_error

from xgboost import XGBRegressor
import lightgbm as lgb
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor, Pool
from sklearn.linear_model import LinearRegression, Ridge, SGDRegressor, HuberRegressor, Lasso


import optuna

import typing

import warnings

%matplotlib inline

In [0]:
import plotly.io as pio
pio.renderers.default = "vscode"

In [0]:
TARGET = "exam_score"
N_SPLITS = 5
SEED = 42
eps = 1e-5

non_relevant_columns = ['id', 'source']

ordinal_maps = {
    "gender" : {"male":0, "female":1, "other":2},
    "internet_access" : {"no":0, "yes":1},
    "sleep_quality" : {"poor":0, "average":1, "good":2},
    "facility_rating" : {"low":0, "medium":1, "high":2},
    "exam_difficulty" : {"easy":0, "moderate":1, "hard":2},
    "course" : {"ba":0, "b.sc":1, "diploma":2, "b.tech":3, "b.com":4, "bca":5, "bba":6},
    "study_method" : {"self-study":0, "online videos":1, "group study":2, "mixed":3, "coaching":4},
}

original_categorical_cols = ['gender', 'course', 'internet_access', 'sleep_quality', 'study_method', 'facility_rating', 'exam_difficulty']


# Data Loader

In [0]:
def load_data(option: str="local"):
  '''
  returns train_df, test_df, df

  '''
  if option == "local":
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')

  elif option == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    train_df = pd.read_csv('/content/drive/MyDrive/Kaggle Practice/Student Scores/train.csv')
    test_df = pd.read_csv('/content/drive/MyDrive/Kaggle Practice/Student Scores/test.csv')

  train_df['source'] = 'train'
  test_df['source'] = 'test'
  df = pd.concat([train_df, test_df], ignore_index=True)

  return train_df, test_df, df

In [0]:
df_train, _, df = load_data(option='local')

# Data Splitter

In [0]:
def data_spliter(df: pd.DataFrame, additional_drop_columns: list = []) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame]:
    '''
    return: X_train, y_train, submission_df

    '''
    train_dataset = df[df['source'] == 'train'].drop(columns=non_relevant_columns + additional_drop_columns)
    submission_dataset = df[df['source'] == 'test'].drop(columns=['source', TARGET] + additional_drop_columns)

    X_train = train_dataset.drop(columns=[TARGET])
    y_train = train_dataset[TARGET]

    return X_train, y_train, submission_dataset

# EDA

In [0]:
df.isna().sum()

In [0]:
df.info()

In [0]:
df.drop(columns=non_relevant_columns).describe(include='all').T

## Correlations

In [0]:
z = df.drop(columns=non_relevant_columns).copy()
z['internet_access'] = z['internet_access'].map({'no':0, 'yes':1})
z['sleep_quality'] = z['sleep_quality'].map({'average':1, 'poor':0, 'good':2})
z['facility_rating'] = z['facility_rating'].map({'medium':1, 'low':0, 'high':2})
z['exam_difficulty'] = z['exam_difficulty'].map({'moderate':1, 'easy':0, 'hard':2})
z = pd.get_dummies(z, columns=['study_method', 'course', 'gender'], dtype=int)

In [0]:
z_corr = z.corr()
z_sorted_cols = z_corr['exam_score'].abs().sort_values(ascending=False).index.to_list()

z_corr_sorted = z_corr.loc[z_sorted_cols, z_sorted_cols]
mask = np.triu(np.ones_like(z_corr_sorted, dtype=bool), k=1)
z_corr_sorted[mask] = np.nan

### Correlation Plot

In [0]:
corr_fig = px.imshow(
    z_corr_sorted,
    text_auto = ".2f",
    aspect=True,
    color_continuous_scale="RdBu_r",
    width=1200,
    height=800
)

corr_fig.show()

## Distribution of categorical columns by exam score

In [0]:
rows = len(original_categorical_cols) // 3 + len(original_categorical_cols) % 3
fig, axes = plt.subplots(rows, 2, figsize=(18, 15))

for ax, c in zip(axes.flat, original_categorical_cols):
    sns.boxplot(data=df, x=c, y="exam_score", ax=ax)
    ax.set_title(c)
    ax.tick_params(axis="x", rotation=45)

# hide any leftover empty panels if you have fewer than 8 columns
for ax in axes.flat[len(original_categorical_cols):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

## Age

In [0]:
ax = sns.countplot(data=df, x='age', color='lightsteelblue')
ax2 = ax.twinx()
sns.pointplot(data=df, x='age', y=TARGET, ax=ax2, color='crimson', errorbar=None)
sns.pointplot(data=df, x='age', y=TARGET, ax=ax2, color='darkorange', errorbar=None, estimator='median', linestyle='--')

## Attendance

In [0]:
plt.figure(figsize=(10,5))
plot_df = df.copy()
plot_df['bins'] = pd.cut(plot_df['class_attendance'], 10, precision=0)
order = plot_df['bins'].unique().sort_values(ascending=True)

ax = sns.countplot(data=plot_df, x='bins', color='lightsteelblue', order=order)
ax2 = ax.twinx()
sns.pointplot(data=plot_df, x='bins', y=TARGET, ax=ax2, color='crimson', errorbar=None)
sns.pointplot(data=plot_df, x='bins', y=TARGET, ax=ax2, color='darkorange', errorbar=None, estimator='median', linestyle='--')

ax.tick_params(axis='x', rotation=45)
plt.tight_layout()

## Course

In [0]:
col = 'course'
order = df.groupby(col)[TARGET].median().sort_values().index
ax = sns.countplot(data=df, x=col, color='lightsteelblue', order=order)
ax2 = ax.twinx()
sns.pointplot(data=df, x=col, y=TARGET, ax=ax2, color='crimson', errorbar=None, label='Mean')
sns.pointplot(data=df, x=col, y=TARGET, ax=ax2, color='darkorange', errorbar=None,
              estimator='median', linestyles='--', markers='s', label='Median')
ax2.legend()

## Sleep Quality

In [0]:
plt.figure(figsize=(8,5))
plot_df = df.copy()
plot_df['sleep_bins'] = pd.cut(df.sleep_hours, 10, precision=0)
stats = plot_df.groupby('sleep_bins')['exam_score'].agg(['mean', 'median'])

order = sorted(plot_df.sleep_bins.unique().tolist())

ax = sns.countplot(plot_df, x='sleep_bins', order=order)
ax2 = ax.twinx()

sns.pointplot(data=plot_df, x='sleep_bins', y=TARGET, ax=ax2, color='crimson', errorbar=None, label='Mean')
sns.pointplot(data=plot_df, x='sleep_bins', y=TARGET, ax=ax2, color='darkorange', errorbar=None,
              estimator='median', linestyles='--', markers='s', label='Median')

plt.tight_layout()

## Study Hours

In [0]:
col = 'study_hours'
bin_col = f'{col}_bins'

plt.figure(figsize=(8,5))
plot_df = df.copy()
plot_df[bin_col] = pd.cut(df[col], 10, precision=0)
stats = plot_df.groupby(bin_col)[TARGET].agg(['mean', 'median'])

order = sorted(plot_df[bin_col].unique().tolist())

ax = sns.countplot(plot_df, x=bin_col, order=order)
ax2 = ax.twinx()

sns.pointplot(data=plot_df, x=bin_col, y=TARGET, ax=ax2,color='crimson', errorbar=None, label='Mean')
sns.pointplot(data=plot_df, x=bin_col, y=TARGET, ax=ax2, color='darkorange',
              estimator='median', linestyles='--', markers='s', label='Median')

plt.tight_layout()

## Study Method

In [0]:
col = 'study_method'
bin_col = col

plt.figure(figsize=(8,5))
plot_df = df.copy()
stats = plot_df.groupby(bin_col)[TARGET].agg(['mean', 'median'])

order = stats.sort_values(by='median').index.tolist()

ax = sns.countplot(plot_df, x=bin_col, order=order)
ax2 = ax.twinx()

sns.pointplot(data=plot_df, x=bin_col, y=TARGET, ax=ax2,color='crimson', errorbar=None, label='Mean')
sns.pointplot(data=plot_df, x=bin_col, y=TARGET, ax=ax2, color='darkorange', errorbar=None,
              estimator='median', linestyles='--', markers='s', label='Median')

plt.tight_layout()

## Internet Acces

In [0]:
col = 'internet_access'
bin_col = col

plt.figure(figsize=(8,5))
plot_df = df.copy()
stats = plot_df.groupby(bin_col)[TARGET].agg(['mean', 'median'])

order = stats.sort_values(by='median').index.tolist()

ax = sns.countplot(plot_df, x=bin_col, order=order)
ax2 = ax.twinx()

sns.pointplot(data=plot_df, x=bin_col, y=TARGET, ax=ax2,color='crimson', label='Mean')
sns.pointplot(data=plot_df, x=bin_col, y=TARGET, ax=ax2, color='darkorange',estimator='median', linestyles='--', markers='s', label='Median')

plt.tight_layout()

# Feature Engineering

In [0]:
df.head()

In [0]:
def feature_engineering(fdf: pd.DataFrame) -> pd.DataFrame:
    print("FEATURE ENGINEERING...")

    df: pd.DataFrame = fdf.copy()

    # Ranking of categories based on liner relationship with target
    for k, i in ordinal_maps.items():
        df[k] = df[k].map(i)

    # Polynomial features
    for pol in df.drop(columns=[TARGET] + non_relevant_columns).select_dtypes(
        include="number"
    ):
        df[f"{pol}_squared"] = df[pol] ** 2

    # Relationship columns
    for col in df.drop(columns=[TARGET] + non_relevant_columns).select_dtypes(
        include="number"
    ):
        relationship_columns: list[str] = ["study_hours", "class_attendance"]
        for r in relationship_columns:
            if col != r:
                df[f"{r}_x_{col}"] = df[r] * df[col]

    # Print which columns were added
    new_columns: list[str] = [
        col
        for col in df.drop(columns=[TARGET] + non_relevant_columns).columns
        if col not in fdf.drop(columns=[TARGET] + non_relevant_columns)
    ]

    print(f"\nFeature Engineering Ended, added {len(new_columns)} columns:")
    for i, c in enumerate(sorted(new_columns)):
        print(f"{i}: {c}")
    return df

In [0]:
_, _, df = load_data()

In [0]:
df = feature_engineering(df)

In [0]:
cv = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

# Ridge Regression

In [0]:
X, y, submission_df = data_spliter(df)
y_bins = pd.cut(y, 10, precision=0, labels=False)

In [0]:
model = make_pipeline(StandardScaler(), Ridge(alpha=10))

oof_ridge = cross_val_predict(model, X, y, cv=cv)
print(f"OOF RMSE: {root_mean_squared_error(y, oof_ridge):.4f}")

In [0]:
model = make_pipeline(StandardScaler(), HuberRegressor(epsilon=10))

oof = cross_val_predict(model, X, y, cv=cv)
print(f"OOF RMSE: {root_mean_squared_error(y, oof):.4f}")

In [0]:
model = make_pipeline(StandardScaler(), Lasso(alpha=10))

oof = cross_val_predict(model, X, y, cv=cv)
print(f"OOF RMSE: {root_mean_squared_error(y, oof):.4f}")

In [0]:
print(root_mean_squared_error(y, oof))

# LightGBM

In [0]:
X, y, submission_df = data_spliter(df)

model.fit(X, y)
submission_df['ridge_pred'] = model.predict(submission_df.drop(columns='id'))
X['ridge_pred'] = oof_ridge
y_bins = pd.cut(y, 10, precision=0, labels=False)

In [0]:
oof_lgb = np.zeros(len(X))
results_lgb = np.zeros(len(submission_df))

X_sub_raw = submission_df.drop(columns='id')

train_rmse_scores, valid_rmse_scores = [], []

best_iterations = []

In [0]:
for fold, (train_idx, val_idx) in enumerate(cv.split(X, y_bins)):

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = LGBMRegressor(
        objective='regression',
        metric='rmse',
        n_estimators=5000,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.7,
        subsample_freq=1,
        colsample_bytree=0.6,
        random_state=SEED,
        verbose=-1,
    )
    model.fit(
        X_train,
        y_train,
        eval_metric="rmse",
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=200),
                   lgb.log_evaluation(period=200)]
    )

    train_prediction = model.predict(X_train)
    validation_prediction = model.predict(X_val)
    oof[val_idx] = validation_prediction
    best_iterations.append(model.best_iteration_)

    train_rmse = root_mean_squared_error(y_train, train_prediction)
    validation_rmse = root_mean_squared_error(y_val, validation_prediction)

    train_rmse_scores.append(train_rmse)
    valid_rmse_scores.append(validation_rmse)

    results_lgb += model.predict(X_sub_raw) / N_SPLITS

    print(f"======= FOLD: {fold + 1} =======")
    print(f"TRAIN RMSE: {train_rmse} | VALID RMSE: {validation_rmse}\n | DELTA: {validation_rmse - train_rmse} | BEST ITER: {model.best_iteration_}")

tr, va = np.array(train_rmse_scores), np.array(valid_rmse_scores)
print("\n====== CROSS VALIDATION ENDED ======")
print(f"TRAIN AVG RMSE: {tr.mean():.5f} +/- {tr.std():.5f} | "
      f"VAL AVG RMSE: {va.mean():.5f} +/- {va.std():.5f}")
print(f"MEAN BEST ITER: {np.mean(best_iterations):.0f}")
print(f"\nOOF RMSE: {root_mean_squared_error(y, oof):.5f}")


In [0]:
submission_df_lgb = submission_df.copy()
submission_df_lgb[TARGET] = results_lgb
submission_df_lgb[['id', TARGET]].to_csv('lgb_submission_v2.csv', index=False)

# XGBoost

In [0]:
X, y, submission_df = data_spliter(df)

y_bins = pd.cut(y, 10, precision=0, labels=False)

In [0]:
oof_xgb = np.zeros(len(X))
results_xgb = np.zeros(len(submission_df))

X_sub_raw = submission_df.drop(columns='id')

train_rmse_scores, valid_rmse_scores = [], []

best_iterations_xgb = []

In [0]:
skf = StratifiedKFold(
    n_splits=N_SPLITS,
    random_state=SEED,
    shuffle=True
)

In [0]:
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_bins)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    model = XGBRegressor(
        objective="reg:squarederror",
        eval_metric="rmse",
        n_estimators=5000,
        learning_rate=0.05,
        min_child_weight=20,
        max_depth=6,
        reg_alpha=3,
        reg_lambda=6,
        colsample_bytree=0.6,
        subsample=0.85,
        early_stopping_rounds=200,
        random_state=SEED
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        verbose=200
    )

    train_prediction = model.predict(X_train)
    validation_prediction = model.predict(X_val)
    oof_xgb[val_idx] = validation_prediction
    best_iterations_xgb.append(model.best_iteration)

    train_rmse = root_mean_squared_error(y_train, train_prediction)
    validation_rmse = root_mean_squared_error(y_val, validation_prediction)

    train_rmse_scores.append(train_rmse)
    valid_rmse_scores.append(validation_rmse)

    results_xgb += model.predict(X_sub_raw) / N_SPLITS

    print(f"======= FOLD: {fold + 1} =======")
    print(f"TRAIN RMSE: {train_rmse} | VALID RMSE: {validation_rmse}\n | DELTA: {validation_rmse - train_rmse} | BEST ITER: {model.best_iteration}")

tr, va = np.array(train_rmse_scores), np.array(valid_rmse_scores)
print("\n====== CROSS VALIDATION ENDED ======")
print(f"TRAIN AVG RMSE: {tr.mean():.5f} +/- {tr.std():.5f} | "
      f"VAL AVG RMSE: {va.mean():.5f} +/- {va.std():.5f}")
print(f"MEAN BEST ITER: {np.mean(best_iterations_xgb):.0f}")
print(f"\nOOF RMSE: {root_mean_squared_error(y, oof_xgb):.5f}")
